# Multi-Resolution Registration with ngff-zarr and ITKElastix

This notebook demonstrates a workflow that combines [ngff-zarr](https://github.com/fideus-labs/ngff-zarr) multi-resolution image pyramids with [ITKElastix](https://github.com/InsightSoftwareConsortium/ITKElastix) registration.

The approach:

1. Load two images and convert them to `ngff_zarr.Multiscales` pyramids with `to_multiscales`.
2. Register the images at a **coarse resolution** for speed (rigid → affine → B-spline).
3. Convert the Elastix result to a standard `itk.CompositeTransform`.
4. Apply the transform to resample the **full-resolution** image in parallel using `dask.array.map_blocks`.

The `ngff_zarr` functions `itk_image_to_ngff_image` and `ngff_image_to_itk_image` bridge between ITK and ngff-zarr data representations, preserving spatial metadata (spacing, origin).

In [ ]:
import itk
import numpy as np
import dask.array as da
import matplotlib.pyplot as plt
from ngff_zarr import (
    itk_image_to_ngff_image,
    ngff_image_to_itk_image,
    to_multiscales,
)

%matplotlib inline

## Load Images

We load the fixed and moving 2D CT head images as ITK float images, which are required by Elastix.

In [ ]:
fixed_image = itk.imread("data/CT_2D_head_fixed.mha")
moving_image = itk.imread("data/CT_2D_head_moving.mha")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=[12, 6])
axs[0].imshow(fixed_image, cmap="gray")
axs[0].set_title("Fixed")
axs[1].imshow(moving_image, cmap="gray")
axs[1].set_title("Moving")
plt.tight_layout()
plt.show()

## Generate Multi-Resolution Pyramids with ngff-zarr

`itk_image_to_ngff_image` converts an `itk.Image` to an `NgffImage`, preserving spacing and origin as `scale` and `translation` metadata. The image data is backed by a dask array.

`to_multiscales` then generates a multi-resolution pyramid. Here we use `scale_factors=[2, 4]`, which produces three levels: the original resolution, 2× downsampled, and 4× downsampled. We will register at the coarsest level for speed.

In [ ]:
ngff_fixed = itk_image_to_ngff_image(fixed_image)
ngff_moving = itk_image_to_ngff_image(moving_image)

scale_factors = [2, 4]
multiscales_fixed = to_multiscales(ngff_fixed, scale_factors=scale_factors)
multiscales_moving = to_multiscales(ngff_moving, scale_factors=scale_factors)

print("Fixed image pyramid:")
for i, image in enumerate(multiscales_fixed.images):
    print(f"  Level {i}: shape={image.data.shape}, scale={image.scale}")

print("\nMoving image pyramid:")
for i, image in enumerate(multiscales_moving.images):
    print(f"  Level {i}: shape={image.data.shape}, scale={image.scale}")

## Register at Coarse Resolution

We extract the coarsest resolution level from both pyramids and convert back to `itk.Image` using `ngff_image_to_itk_image`. We pass `wasm=False` to get native `itk.Image` objects, which are required by ITKElastix.

We then run a three-stage Elastix registration (rigid → affine → B-spline). Registering at lower resolution is significantly faster. Because the resulting transform is defined in physical coordinates, it can be applied at any resolution level.

In [ ]:
# Extract the coarsest resolution level
fixed_coarse_ngff = multiscales_fixed.images[-1]
moving_coarse_ngff = multiscales_moving.images[-1]

print(f"Coarse fixed:  shape={fixed_coarse_ngff.data.shape}, scale={fixed_coarse_ngff.scale}")
print(f"Coarse moving: shape={moving_coarse_ngff.data.shape}, scale={moving_coarse_ngff.scale}")

In [ ]:
# Convert NgffImage back to itk.Image for Elastix
fixed_coarse = ngff_image_to_itk_image(fixed_coarse_ngff, wasm=False)
moving_coarse = ngff_image_to_itk_image(moving_coarse_ngff, wasm=False)

# Ensure float32 pixel type as required by Elastix
ImageType = itk.Image[itk.F, 2]
if not isinstance(fixed_coarse, ImageType):
    fixed_coarse = itk.cast_image_filter(fixed_coarse, ttype=(type(fixed_coarse), ImageType))
if not isinstance(moving_coarse, ImageType):
    moving_coarse = itk.cast_image_filter(moving_coarse, ttype=(type(moving_coarse), ImageType))

In [ ]:
# Set up multi-stage registration: rigid -> affine -> bspline
parameter_object = itk.ParameterObject.New()
parameter_object.AddParameterMap(
    itk.ParameterObject.GetDefaultParameterMap("rigid")
)
parameter_object.AddParameterMap(
    itk.ParameterObject.GetDefaultParameterMap("affine")
)
parameter_object.AddParameterMap(
    itk.ParameterObject.GetDefaultParameterMap("bspline")
)

In [ ]:
# Run registration using the object-oriented API.
# We need the registration_method object to later extract the ITK transform.
registration_method = itk.ElastixRegistrationMethod[
    type(fixed_coarse), type(moving_coarse)
].New(
    fixed_image=fixed_coarse,
    moving_image=moving_coarse,
    parameter_object=parameter_object,
)
registration_method.SetLogToConsole(False)
registration_method.Update()

result_image_coarse = registration_method.GetOutput()

In [ ]:
# Visualize the coarse registration result
fig, axs = plt.subplots(1, 3, figsize=[15, 5])
axs[0].imshow(fixed_coarse, cmap="gray")
axs[0].set_title("Fixed (Coarse)")
axs[1].imshow(moving_coarse, cmap="gray")
axs[1].set_title("Moving (Coarse)")
axs[2].imshow(result_image_coarse, cmap="gray")
axs[2].set_title("Registered (Coarse)")
plt.tight_layout()
plt.show()

## Convert to ITK Transform

The `ConvertToItkTransform` method converts the Elastix registration results into a standard `itk.CompositeTransform`. This composite transform chains the rigid, affine, and B-spline stages into a single object that can be used with any ITK resampling filter.

Because the transform is defined in **physical coordinates** (not pixel indices), it applies correctly at any resolution level.

In [ ]:
elx_advanced_transform = registration_method.GetCombinationTransform()
itk_composite_transform = itk.CompositeTransform[itk.D, 2].cast(
    registration_method.ConvertToItkTransform(elx_advanced_transform)
)
print(itk_composite_transform)

## Apply Transform at Full Resolution with dask.array.map_blocks

Now we apply the transform obtained at coarse resolution to resample the **full-resolution** moving image. We use `dask.array.map_blocks` to process each chunk of the output image independently and in parallel.

The strategy:
- The full-resolution fixed image's dask array serves as the **output grid template**. Each chunk defines a spatial region in the output.
- For each chunk, we use `block_info` to compute its physical origin from the array location and the image's spacing and origin metadata.
- We create a small ITK reference image for that chunk's spatial region and call `itk.resample_image_filter` to resample the moving image into it.

For simplicity, the full moving image is loaded into memory so that each block function can sample from it. More memory-efficient approaches (e.g., lazy region-based reads) will be available in future ngff-zarr releases.

In [ ]:
# Get full-resolution NgffImage data (dask-backed)
fixed_full = multiscales_fixed.images[0]
moving_full = multiscales_moving.images[0]

# Materialize the full-resolution moving image as an itk.Image for resampling.
# Each block function will sample from this image.
moving_itk_full = ngff_image_to_itk_image(moving_full, wasm=False)
if not isinstance(moving_itk_full, ImageType):
    moving_itk_full = itk.cast_image_filter(
        moving_itk_full, ttype=(type(moving_itk_full), ImageType)
    )

print(f"Full-resolution fixed:  shape={fixed_full.data.shape}")
print(f"Full-resolution moving: shape={moving_full.data.shape}")

In [ ]:
def resample_block(block, block_info=None, transform=None, moving_image=None,
                   spacing_itk=None, origin_itk=None):
    """Resample a single output block from the moving image using the given transform.

    Parameters
    ----------
    block : numpy array
        The chunk from the fixed image grid (used only for its shape).
    block_info : dict
        Provided by dask.array.map_blocks. Contains chunk position information.
    transform : itk.Transform
        The spatial transform mapping fixed to moving image space.
    moving_image : itk.Image
        The full-resolution moving image to resample from.
    spacing_itk : list
        Pixel spacing in ITK order [x, y].
    origin_itk : list
        Image origin in ITK order [x, y].
    """
    if block_info is None:
        return block

    # block_info[0]['array-location'] gives [(y_start, y_stop), (x_start, x_stop)]
    # for a 2D array with dims (y, x)
    array_location = block_info[0]["array-location"]

    # Compute the physical origin for this block.
    # array-location is in (y, x) order; ITK origin is in (x, y) order.
    block_origin_itk = [
        origin_itk[0] + array_location[1][0] * spacing_itk[0],  # x
        origin_itk[1] + array_location[0][0] * spacing_itk[1],  # y
    ]

    block_shape = block.shape  # (rows, cols) = (y_size, x_size)

    # Create a reference image for this block's spatial region
    reference = itk.Image[itk.F, 2].New()
    region = itk.ImageRegion[2]()
    region.SetSize([int(block_shape[1]), int(block_shape[0])])  # ITK size is [x, y]
    region.SetIndex([0, 0])
    reference.SetRegions(region)
    reference.SetSpacing(spacing_itk)
    reference.SetOrigin(block_origin_itk)
    reference.Allocate()

    # Resample the moving image into this block's region
    resampled = itk.resample_image_filter(
        moving_image,
        transform=transform,
        use_reference_image=True,
        reference_image=reference,
    )

    return itk.array_from_image(resampled)

In [ ]:
# Extract spacing and origin from the full-resolution fixed image.
# NgffImage scale/translation use dim-name keys in (y, x) order.
# ITK uses (x, y) order.
spacing_itk = [fixed_full.scale["x"], fixed_full.scale["y"]]
origin_itk = [fixed_full.translation["x"], fixed_full.translation["y"]]

# Apply the transform in parallel across all chunks
resampled_dask = da.map_blocks(
    resample_block,
    fixed_full.data,
    dtype=np.float32,
    transform=itk_composite_transform,
    moving_image=moving_itk_full,
    spacing_itk=spacing_itk,
    origin_itk=origin_itk,
)

print(f"Resampled dask array: shape={resampled_dask.shape}, chunks={resampled_dask.chunks}")

In [ ]:
# Compute the result (triggers parallel execution across chunks)
resampled_array = resampled_dask.compute()
print(f"Resampled result shape: {resampled_array.shape}")

## Visualize Results

Compare the full-resolution resampled result with the fixed and moving images.

In [ ]:
fixed_array = np.asarray(fixed_full.data)

fig, axs = plt.subplots(1, 4, figsize=[20, 5])
axs[0].imshow(fixed_array, cmap="gray")
axs[0].set_title("Fixed (Full Resolution)")
axs[1].imshow(np.asarray(moving_full.data), cmap="gray")
axs[1].set_title("Moving (Full Resolution)")
axs[2].imshow(resampled_array, cmap="gray")
axs[2].set_title("Resampled Moving")
diff = fixed_array - resampled_array
im = axs[3].imshow(diff, cmap="RdBu")
axs[3].set_title("Difference")
fig.colorbar(im, ax=axs[3], orientation="vertical", fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## Summary

This example demonstrated a multi-resolution registration and resampling workflow:

- **`ngff_zarr.itk_image_to_ngff_image`** and **`ngff_zarr.ngff_image_to_itk_image`** bridge between ITK and ngff-zarr data representations, preserving spatial metadata.
- **`ngff_zarr.to_multiscales`** generates multi-resolution pyramids from a single image with one call.
- **Coarse-resolution registration** with ITKElastix (rigid → affine → B-spline) is fast, and because the resulting transform is defined in physical coordinates, it applies at any resolution.
- **`dask.array.map_blocks`** enables parallel resampling at full resolution, processing each output chunk independently with `itk.resample_image_filter`.